In [ ]:
#Part A: Data inspection and column selection
#Cell 1 – Import libraries and load dataset
import pandas as pd
import numpy as np

RAW_PATH = "data/accepted_2007_to_2018Q4.csv"

# The file has 2,260,701 data rows (not counting the header line)
TOTAL_ROWS = 2260701
LAST_ROWS = 100000

# Skip the data lines from the top, but keep line 0 (the column names)
skip_lines = range(1, TOTAL_ROWS - LAST_ROWS + 1)

# Read only the last 1 lakh rows for initial inspection
df_all = pd.read_csv(
    RAW_PATH,
    skiprows=skip_lines,
    low_memory=False
)

print("Rows and columns:", df_all.shape)

Rows and columns: (100000, 151)


In [205]:
#Cell 2 – Select 24 columns to keep
# Keep only features available when the loan is issued
keep_cols = [
    # Target and date
    "loan_status",
    "issue_d",

    # Loan details
    "loan_amnt",
    "term",
    "int_rate",
    "grade",
    "purpose",

    # Customer information
    "emp_length",
    "home_ownership",
    "annual_inc",
    "addr_state",

    # Credit history
    "dti",
    "fico_range_low",
    "fico_range_high",
    "earliest_cr_line",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "total_acc",
    "pub_rec",
    "pub_rec_bankruptcies",
    "revol_bal",
    "revol_util",
    "mort_acc"
]

print("Columns we KEEP:", len(keep_cols))

Columns we KEEP: 24


In [206]:
#Cell 3 – Remove IDs and free-text columns
# IDs do not provide useful credit risk information
# Free-text columns are excluded to keep modelling simple

id_text_cols = [
    "id",
    "member_id",
    "url",
    "desc",
    "emp_title",
    "title",
    "zip_code"
]

In [207]:
#Cell 4 – Remove payment leakage columns
# These columns contain information recorded after loan issuance
# They must not be used for predicting default at loan origination

leak_payment_cols = [
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low",
    "pymnt_plan"
]

In [208]:
#Cell 5 – Remove hardship and settlement leakage
# Hardship information becomes available after repayment problems occur
leak_hardship_cols = [
    "hardship_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "deferral_term",
    "hardship_amount",
    "hardship_start_date",
    "hardship_end_date",
    "payment_plan_start_date",
    "hardship_length",
    "hardship_dpd",
    "hardship_loan_status",
    "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount",
    "hardship_last_payment_amount"
]

# Settlement information is also future information

leak_settlement_cols = [
    "debt_settlement_flag",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term"
]


In [209]:
#Cell 6 – Remove duplicate, joint and constant columns
# Duplicate loan amount information
duplicate_cols = [
    "funded_amnt",
    "funded_amnt_inv"
]

# Joint applicant information
joint_cols = [
    "annual_inc_joint",
    "dti_joint",
    "verification_status_joint",
    "revol_bal_joint",
    "sec_app_fico_range_low",
    "sec_app_fico_range_high",
    "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths",
    "sec_app_mort_acc",
    "sec_app_open_acc",
    "sec_app_revol_util",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_mths_since_last_major_derog"
]

# Constant column
constant_cols = [
    "policy_code"
]

In [210]:
#Cell 7 – Check all column lists
# Combine all column lists
listed = (
    keep_cols +
    id_text_cols +
    leak_payment_cols +
    leak_hardship_cols +
    leak_settlement_cols +
    duplicate_cols +
    joint_cols +
    constant_cols
)

# Check whether all columns exist in the dataset
for col in listed:
    if col not in df_all.columns:
        print("NOT FOUND:", col)

# Find additional columns not included in our lists
extra_cols = [
    col for col in df_all.columns
    if col not in listed
]

print("Extra credit-report columns:", len(extra_cols))

Extra credit-report columns: 63


In [211]:
#Cell 8 – Summary of columns
summary = pd.DataFrame({
    "Group": [
        "KEEP",
        "LEAVE: IDs and free text",
        "LEAVE: Payment leakage",
        "LEAVE: Hardship leakage",
        "LEAVE: Settlement leakage",
        "LEAVE: Duplicate columns",
        "LEAVE: Joint applications",
        "LEAVE: Constant column",
        "LEAVE: Extra columns"
    ],

    "Columns": [
        len(keep_cols),
        len(id_text_cols),
        len(leak_payment_cols),
        len(leak_hardship_cols),
        len(leak_settlement_cols),
        len(duplicate_cols),
        len(joint_cols),
        len(constant_cols),
        len(extra_cols)
    ]
})

print("Total columns:", df_all.shape[1])
print("Columns we KEEP:", len(keep_cols))
print("Columns we LEAVE:", df_all.shape[1] - len(keep_cols))

display(summary)

Total columns: 151
Columns we KEEP: 24
Columns we LEAVE: 127


,Group,Columns
0,KEEP,24
1,LEAVE: IDs and free text,7
2,LEAVE: Payment leakage,16
3,LEAVE: Hardship leakage,15
4,LEAVE: Settlement leakage,7
5,LEAVE: Duplicate columns,2
6,LEAVE: Joint applications,16
7,LEAVE: Constant column,1
8,LEAVE: Extra columns,63


In [212]:
#Cell 9 – Example of data leakage
# fico_range_low = credit score at loan application
# last_fico_range_low = credit score later in the loan lifecycle
# The second column must not be used for origination prediction

df_all[
    [
        "fico_range_low",
        "last_fico_range_low",
        "loan_status"
    ]
].head(10)

,fico_range_low,last_fico_range_low,loan_status
0,730.0,705.0,Current
1,775.0,715.0,Fully Paid
2,790.0,645.0,Late (31-120 days)
3,695.0,640.0,Current
4,780.0,740.0,Current
5,705.0,690.0,Fully Paid
6,770.0,750.0,Fully Paid
7,695.0,710.0,Current
8,720.0,690.0,Fully Paid
9,745.0,710.0,Fully Paid


In [213]:
#Part B: Data cleaning and preparation
#Cell 10 – Load selected columns from full CSV
# Load only the 24 selected columns
# This reduces memory usage

df = pd.read_csv(
    RAW_PATH,
    usecols=keep_cols,
    low_memory=False
)

print("Rows and columns:", df.shape)

Rows and columns: (2260701, 24)


In [214]:
# Check the different loan outcomes

print(df["loan_status"].value_counts())

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [215]:
#Cell 12 – Create target variable
# Keep loans with known outcomes
df = df[
    df["loan_status"].isin([
        "Fully Paid",
        "Charged Off",
        "Default"
    ])
].copy()

# Fully Paid = 0
# Charged Off / Default = 1

df["default"] = (
    df["loan_status"] != "Fully Paid"
).astype(int)

print("Rows:", len(df))

print(
    "Default rate:",
    round(df["default"].mean() * 100, 2),
    "%"
)

Rows: 1345350
Default rate: 19.96 %


In [216]:
#Cell 13 – Convert text into numbers
# Convert term from "36 months" to 36
df["term"] = (
    df["term"]
    .str.replace("months", "", regex=False)
    .str.strip()
)

df["term"] = pd.to_numeric(
    df["term"],
    errors="coerce"
)

# Convert employment length into years

emp_map = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10
}

df["emp_length"] = df["emp_length"].map(emp_map)

df[["term", "emp_length"]].head()

,term,emp_length
0,36,10.0
1,36,10.0
2,60,10.0
4,60,3.0
5,36,4.0


In [217]:

#Cell 14 – Convert dates into years
# Convert loan issue date into year
df["issue_year"] = pd.to_datetime(
    df["issue_d"],
    format="%b-%Y",
    errors="coerce"
).dt.year

# Convert earliest credit line into year
df["earliest_year"] = pd.to_datetime(
    df["earliest_cr_line"],
    format="%b-%Y",
    errors="coerce"
).dt.year

df[["issue_d", "issue_year"]].head()

,issue_d,issue_year
0,Dec-2015,2015
1,Dec-2015,2015
2,Dec-2015,2015
4,Dec-2015,2015
5,Dec-2015,2015


In [218]:
#Cell 15 – Create new useful features
# Calculate average FICO credit score
df["fico"] = (
    df["fico_range_low"] +
    df["fico_range_high"]
) / 2

# Calculate credit history length
df["credit_history_years"] = (
    df["issue_year"] -
    df["earliest_year"]
)

# Remove rows where the first credit account is AFTER the loan (impossible, so the data is wrong)
# Rows with an empty value are kept, they are filled later in notebook 03
df = df[(df["credit_history_years"].isna()) | (df["credit_history_years"] >= 0)].copy()

# Calculate loan amount compared to annual income
df["loan_to_income"] = (
    df["loan_amnt"] /
    df["annual_inc"].replace(0, np.nan)
)

# Remove original columns that are no longer required
df = df.drop(columns=[
    "loan_status",
    "fico_range_low",
    "fico_range_high",
    "earliest_cr_line",
    "earliest_year",
    "issue_d"
])

print("Rows left:", len(df))
print(df.columns.tolist())

Rows left: 1345350
['loan_amnt', 'term', 'int_rate', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'default', 'issue_year', 'fico', 'credit_history_years', 'loan_to_income']


In [219]:
#Cell 16 – Remove clearly wrong values
# Store row count before cleaning
before = len(df)

# Annual income must be greater than zero
df = df[df["annual_inc"] > 0]

# DTI must be between 0 and 100
df = df[
    (df["dti"].isna()) |
    ((df["dti"] >= 0) & (df["dti"] <= 100))
]

print("Rows removed:", before - len(df))
print("Rows left:", len(df))

Rows removed: 896
Rows left: 1344454


In [220]:
#Cell 17 – Keep loans issued up to 2015
# Keep older loans to reduce incomplete loan outcome bias
df = df[df["issue_year"] <= 2015]

print("Rows:", len(df))

print(
    "Default rate:",
    round(df["default"].mean() * 100, 2),
    "%"
)

print(
    df["issue_year"].value_counts().sort_index()
)

Rows: 826596
Default rate: 18.43 %
issue_year
2007       251
2008      1562
2009      4716
2010     11536
2011     21721
2012     53367
2013    134804
2014    223103
2015    375536
Name: count, dtype: int64


In [221]:
#Cell 18 – Check missing values
# Calculate missing values and percentages
# Do not fill missing values yet
# Imputation will happen after train/test splitting

missing = (
    df.isna().mean() * 100
).round(2).sort_values(ascending=False)

display(missing[missing > 0])

mort_acc                5.72
emp_length              5.15
pub_rec_bankruptcies    0.08
revol_util              0.05
dtype: float64

In [222]:
#Cell 19 – Take random sample of 1 lakh rows
# Select 100,000 rows for modelling
SAMPLE_SIZE = 100000

if len(df) > SAMPLE_SIZE:
    df = df.sample(
        n=SAMPLE_SIZE,
        random_state=42
    )

# Sort by issue year
df = df.sort_values("issue_year")

# Reset index
df = df.reset_index(drop=True)

print("Final rows:", len(df))

print(
    "Default rate:",
    round(df["default"].mean() * 100, 2),
    "%"
)

Final rows: 100000
Default rate: 18.46 %


In [223]:
#Cell 20 – Save cleaned dataset
# Save the cleaned dataset in Parquet format

df.to_parquet(
    "data/loans_clean.parquet",
    index=False
)

print("Clean dataset saved successfully!")

print("Final dataset shape:", df.shape)

display(df.head())

Clean dataset saved successfully!
Final dataset shape: (100000, 24)


,loan_amnt,term,int_rate,grade,emp_length,home_ownership,annual_inc,purpose,addr_state,dti,...,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,default,issue_year,fico,credit_history_years,loan_to_income
0,2225.0,36,10.59,C,2.0,RENT,60000.0,moving,CA,8.14,...,20511.0,62.0,9.0,NaN,NaN,1,2007,707.0,13,0.037083
1,14400.0,36,12.80,D,1.0,RENT,36120.0,debt_consolidation,NE,8.47,...,7104.0,93.5,25.0,NaN,NaN,1,2007,662.0,7,0.398671
2,4925.0,36,9.96,B,10.0,MORTGAGE,84000.0,debt_consolidation,VA,4.84,...,48251.0,50.1,28.0,NaN,NaN,0,2007,727.0,17,0.058631
3,11300.0,36,12.17,D,8.0,RENT,33600.0,debt_consolidation,IN,23.96,...,12102.0,49.0,15.0,NaN,NaN,0,2007,712.0,8,0.336310
4,15000.0,36,10.28,C,0.0,RENT,88000.0,credit_card,WA,11.95,...,24615.0,94.7,6.0,NaN,NaN,0,2007,702.0,14,0.170455
